# 5 pamoka – Agentinis RAG


## Nustatymas

Šiame užrašų knygelėje demonstruojamas Agentic RAG (paieška papildyto generavimo) šablonas naudojant Microsoft Agent Framework.

**Išankstiniai reikalavimai:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — jūsų Azure AI Search paslaugos galinis taškas
- `AZURE_SEARCH_API_KEY` — jūsų Azure AI Search API raktas
- Azure OpenAI diegimas sukonfigūruotas naudojant aplinkos kintamuosius
- Azure CLI patvirtintas (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kas yra Agentic RAG?

Tradicinis RAG vykdo fiksuotą procesą: pirmiausia renkasi dokumentus, tada generuoja atsakymą. **Agentic RAG** žengia toliau suteikdamas agentui autonomiją nuspręsti **kada** ir **kaip** gauti informaciją.

Naudodamas Agentic RAG, agentas gali:
- **Nuspręsti**, ar reikia rinkti informaciją prieš atsakant į klausimą
- **Pasirinkti**, kurį duomenų šaltinį ar įrankį užklausti
- **Įvertinti** gautus rezultatus ir atlikti papildomą rinkimą, jei pirmasis bandymas buvo nepakankamas
- **Sujungti** informaciją iš kelių rinkimo etapų į nuoseklų atsakymą

Tai daro agentą lankstesnį ir tikslesnį, palyginti su statiniu rink ir po to generuok procesu.


## Paieškos įrankio kūrimas

Agentic RAG išoriniai duomenų šaltiniai yra apgaubti kaip **įrankiai**, kuriuos agentas gali iškviesti pagal poreikį. Tai leidžia agentui traktuoti paiešką kaip dar vieną veiksmą, kurį jis gali atlikti, o ne privalomą žingsnį.

Toliau apibrėžiame kelionės žinių bazę ir pateikiame ją kaip įrankį, kurį agentas gali iškviesti, norėdamas sužinoti informacijos apie kelionės tikslus.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG agento kūrimas

Dabar sukursime agentą, kuris yra nurodytas **visada gauti informaciją prieš atsakant**. Agentas naudoja įrankį `search_travel_knowledge`, kad pagrįstų savo atsakymus žinių baze, o ne pasikliautų savo mokymo duomenimis.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratyvus paieškos procesas — Maker-Checker modelis

Svarbiausia Agentic RAG pranašumas yra **iteratyvi paieška**. Agentas gali atlikti kelis paieškos etapus, kad patikrintų, patobulintų ar išplėstų savo pradinius radinius — panašiai kaip „maker-checker“ darbo eiga:

1. **Maker žingsnis**: Agentas surenka pradinę informaciją ir parengia atsakymą.
2. **Checker žingsnis**: Agentas atlieka papildomą paiešką, kad patikrintų detales ar užpildytų spragas.

Žemiau agentui pateiktas klausimas, reikalaujantis palyginti kelias paskirties vietas, todėl jis atlieka keletą paieškų.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Santrauka

Šioje pamokoje išmokote, kaip sukurti **Agentinę RAG** sistemą, naudojant Microsoft Agent Framework:

- **Agentinė RAG** leidžia agentams savarankiškai spręsti, kada vykdyti informacijos paiešką, todėl paieška tampa dinamiška, o ne fiksuota.
- **Įrankiai kaip duomenų šaltiniai**: Išorinės žinių bazės (pvz., Azure AI Search) yra apgaubtos kaip įrankiai, kuriuos agentas gali paleisti.
- **Iteratyvi paieška**: maker-checker modelis leidžia agentui atlikti kelis paieškos etapus — ieškoti, tikrinti ir patikslinti — prieš pateikiant galutinį atsakymą.

Gaminiame vietoje atminties `TRAVEL_KNOWLEDGE_BASE` naudotumėte tikrą Azure AI Search indeksą, kad galėtumėte tvarkyti didelio masto kelionių dokumentų paiešką.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
